# Table 6 — Heterogeneous Effects of Inflation

Replication code for the heterogeneity regressions from:

> Albertazzi, U., 't Hooft, J., & ter Steege, L. (2025).  
> *The causal effect of inflation on financial stability: evidence from history.*  
> ECB Working Paper Series No. 3108.

---

## Methodology

Table 6 extends the second-stage regression from Equation 4 by interacting
instrumented inflation $\widehat{\Delta\pi}_{i,t-1}$ with country-level
heterogeneity variables. Each column tests a different transmission channel
for the redistribution hypothesis — the idea that inflation erodes real household
income, weakening debt serviceability and raising crisis risk:

| Column | Heterogeneity variable | Channel tested |
|---|---|---|
| 1 | Wage growth | Inflation persistence: do wages catch up? |
| 2 | Mortgage, corporate and public debt ratios | Balance sheet vulnerability |
| 3 | CB financial independence | Credibility anchor for inflation expectations |

All columns share the same first stage (Equation 3): oil supply shocks instrument
annual inflation. Controls are contemporaneous and four lags of short-term interest
rate changes and real GDP growth. Country fixed effects throughout.
Sample: 1975–2020.

## 1. Imports

In [33]:
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
from linearmodels.panel import PanelOLS
import statsmodels.api as sm

## 2. Helper Functions

Small utilities used throughout the notebook.

In [34]:
# ── Data reshaping ─────────────────────────────────────────────────────────────

def pivot(variable):
    """Wide format: rows = years, columns = countries."""
    return raw_df.pivot(index='year', columns='country', values=variable)

def melt(df):
    """Long format suitable for a (country, year) MultiIndex panel."""
    return df.melt(ignore_index=False).reset_index().set_index(['country', 'year'])

def add_const(df):
    """Prepend a constant column (required by linearmodels)."""
    return sm.add_constant(df)

def winsor(df):
    """Winsorise at the 5th and 95th percentiles to limit outlier influence."""
    return df.apply(lambda x: winsorize(x, limits=(.05, .05)))

### Regression utilities

Standard display helpers plus a shared runner for all three heterogeneity specs.

In [35]:
def significance_stars(pval):
    """Return AER-style significance stars for a given p-value."""
    if pval < 0.01:
        return '***'
    elif pval < 0.05:
        return '**'
    elif pval < 0.1:
        return '*'
    return ''


def display(fit):
    """Coefficients with significance stars only."""
    df = pd.DataFrame({'coefficient': fit.params, 'pvalue': fit.pvalues})
    df['significance'] = df['pvalue'].apply(significance_stars)
    df['results'] = df.apply(
        lambda x: f"{x['coefficient']:.3f}{x['significance']}", axis=1)
    return df[['results']]


def display_w_std_errors(fit):
    """Coefficients with significance stars and clustered standard errors."""
    df = pd.DataFrame({
        'coefficient': fit.params,
        'pvalue':      fit.pvalues,
        'std_error':   round(fit.std_errors, 3),
    })
    df['significance'] = df['pvalue'].apply(significance_stars)
    df['results'] = df.apply(
        lambda x: f"{x['coefficient']:.3f}{x['significance']}", axis=1)
    df['results_w_errors'] = (
        df['results'] + ' (' + df['std_error'].astype(str) + ')')
    return df[['results_w_errors']]


def run_heterogeneity_regression(extra_vars):
    """
    Run a second-stage heterogeneity regression.
    
    Builds a spec of delta_inflation_hat + extra_vars + covariates and fits
    a country-FE linear probability model with clustered standard errors.
    Returns the key rows (constant through the last extra variable), dropping
    the control lags from the display.
    
    Parameters
    ----------
    extra_vars : list of str
        Column names already in panel_df: typically the heterogeneity level
        variable and its interaction with delta_inflation_hat.
        The order determines the display order — put the interaction first.
    
    Returns
    -------
    pd.DataFrame
        Coefficient table with standard errors for the key rows only.
    """
    spec = ['delta_inflation_hat'] + extra_vars + covariates
    df   = panel_df[['crisis'] + spec].dropna()
    fit  = PanelOLS(
        dependent      = df['crisis'],
        exog           = add_const(df[spec]),
        entity_effects = True,
        time_effects   = False,
    ).fit(cov_type='clustered', cluster_entity=True)
    # Show const + delta_inflation_hat + extra_vars; hide the control lags
    n_key = 1 + 1 + len(extra_vars)  # const, delta_inflation_hat, extra
    return display_w_std_errors(fit).iloc[:n_key]

## 3. Data

### 3.1 Jordà-Schularick-Taylor Macrohistory Database

Annual macro-financial data for 18 advanced economies, 1870–2020.  
Source: Jordà, Schularick & Taylor (2017); dataset R6 available at
[www.macrohistory.net/database](https://www.macrohistory.net/database/).

In [36]:
raw_df = (
    pd.read_excel('datasets/JSTdatasetR6.xlsx')
    .drop(columns=['country', 'ifs'])
    .rename(columns={'iso': 'country'})
)

### 3.2 Baumeister-Hamilton Oil Supply Shocks

Monthly structural oil supply shocks, aggregated to annual frequency.  
Source: Baumeister & Hamilton (2019).

In [37]:
raw_oil_shocks_df = pd.read_excel('datasets/BH2_supply_shocks.xlsx', header=1, usecols=[1])

# Assign monthly dates starting February 1975
raw_oil_shocks_df.index = pd.date_range(
    start='1975-02-01', periods=len(raw_oil_shocks_df), freq='ME')

# Pad with a January 1975 NaN row so the annual mean covers the full first year
raw_oil_shocks_df = raw_oil_shocks_df.reindex(
    pd.date_range(start='1975-01-01', periods=len(raw_oil_shocks_df) + 1, freq='ME'))

# Resample to annual (calendar-year mean × 12)
oil_shocks = raw_oil_shocks_df.resample('YE').mean() * 12
oil_shocks.index = range(1975, 1975 + len(oil_shocks))

### 3.3 Romelli Central Bank Independence Index

The `cbie_finindep` sub-index measures the degree to which the central bank is
financially independent of the government — a key dimension of credibility.  
Source: Romelli (2024); available at [www.damianoromelli.com](https://www.damianoromelli.com/).

In [38]:
credibility_raw_df = pd.read_excel(
    'datasets/CBIData_Romelli_2024.xlsx',
    sheet_name=1,
    usecols=['year', 'iso_a3', 'cbie_finindep'],
).rename(columns={'iso_a3': 'country'})

# Merge into the main dataset
raw_df = pd.merge(raw_df, credibility_raw_df, how='left', on=['year', 'country'])

## 4. Variable Construction

In [39]:
# ── Core macro series ──────────────────────────────────────────────────────────
cpi_df    = pivot('cpi')        # consumer price index (level)
gdp_df    = pivot('rgdpbarro')  # real GDP per capita (Barro-Ursúa)
stir_df   = pivot('stir')       # short-term interest rate (level)
crisis_df = pivot('crisisJST')  # systemic banking crisis dummy

# ── Instrument: oil supply shock broadcast to all countries ───────────────────
oil_shocks_df = cpi_df.copy()
oil_shocks_df[:] = np.nan
for country in oil_shocks_df.columns:
    oil_shocks_df[country] = oil_shocks['oil supply shocks']

# ── Heterogeneity variables ────────────────────────────────────────────────────

# Column 1: wage growth (annual, winsorised) — proxy for real income recovery
wages_df = winsor(pivot('wage').pct_change(fill_method=None) * 100)

# Column 2: debt ratios (winsorised)
mortgage_to_gdp_df     = winsor(pivot('tmort') / pivot('gdp'))
corporate_debt_to_gdp_df = winsor(pivot('bdebt') / pivot('gdp'))
public_debt_df         = pivot('debtgdp')

# Column 3: CB financial independence, demeaned by cross-sectional median
# A positive value means the CB is more independent than the sample median
# in that year; the interaction tests whether higher independence attenuates
# the inflation-crisis link.
cb_fin_indep_df = pivot('cbie_finindep')
cb_fin_indep_relative_df = cb_fin_indep_df.sub(cb_fin_indep_df.median(axis=1), axis=0)

## 5. Regression Panel

The panel is built once and shared across all three heterogeneity regressions.
Heterogeneity levels are included here; their interactions with instrumented
inflation are added after the first stage (Section 7), once fitted values are available.

In [40]:
panel_df = raw_df.set_index(['country', 'year'])
panel_df['year'] = panel_df.reset_index()['year'].values

# ── Dependent variable ────────────────────────────────────────────────────────
# Equation 4 uses the raw crisis onset dummy (not the 3-year window).
panel_df['crisis'] = melt(crisis_df)

# ── Endogenous variable and instrument ────────────────────────────────────────
panel_df['infl']      = melt(winsor((cpi_df.pct_change() * 100).diff()))
panel_df['oil_shock'] = melt(oil_shocks_df)

# ── Controls: lags 0–4 of interest rate changes and GDP growth ────────────────
covariates = []
for lag in range(4 + 1):
    panel_df[f'stir(-{lag})'] = melt(stir_df.diff().shift(lag))
    panel_df[f'gdp(-{lag})']  = melt((gdp_df.pct_change(fill_method=None) * 100).shift(lag))
    covariates.append(f'stir(-{lag})')
    covariates.append(f'gdp(-{lag})')

# ── Heterogeneity levels (lagged 2 years to reduce reverse-causality risk) ─────
panel_df['wages']            = melt(wages_df.shift(2))
panel_df['mortgages']        = melt(mortgage_to_gdp_df.shift(2))
panel_df['corporate_debt']   = melt(corporate_debt_to_gdp_df.shift(2))
panel_df['public_debt']      = melt(public_debt_df.shift(2))
panel_df['cb_fin_indep']     = melt(cb_fin_indep_relative_df.shift(2))

## 6. First Stage — Equation 3

The first stage is common to all three heterogeneity columns.
A negative coefficient on `oil_shock` confirms the instrument is relevant:
a negative (adverse) oil supply shock raises inflation.

In [41]:
first_stage_covariates = ['oil_shock'] + covariates

first_stage_df = panel_df[['infl'] + first_stage_covariates].dropna()

first_stage_fit = PanelOLS(
    dependent      = first_stage_df['infl'],
    exog           = add_const(first_stage_df[first_stage_covariates]),
    entity_effects = True,
    time_effects   = False,
).fit(cov_type='clustered', cluster_entity=True)

# Extract fitted values and pivot back to wide (year × country) format
fitted_df = (
    first_stage_fit.fitted_values
    .reset_index()
    .pivot(index='year', columns='country', values='fitted_values')
)

display(first_stage_fit).iloc[:2]  # oil_shock coefficient

,results
const,-0.493***
oil_shock,-0.060***


## 7. Interaction Terms

With fitted values in hand, we construct the interaction of instrumented inflation
$(\widehat{\Delta\pi}_{i,t-1})$ with each heterogeneity variable.
A negative interaction coefficient means the variable *attenuates* the baseline
inflation-crisis link; a positive one means *amplification*.

In [42]:
# Instrumented inflation, lagged one year (the second-stage regressor of interest)
infl_hat_1 = fitted_df.shift(1)
panel_df['delta_inflation_hat'] = melt(infl_hat_1)

# Column 1: wage growth interaction
panel_df['infl_hat X wages']         = melt(infl_hat_1 * wages_df.shift(2))

# Column 2: debt interactions
panel_df['infl_hat X mortgages']     = melt(infl_hat_1 * mortgage_to_gdp_df.shift(2))
panel_df['infl_hat X corporate_debt']= melt(infl_hat_1 * corporate_debt_to_gdp_df.shift(2))
panel_df['infl_hat X public_debt']   = melt(infl_hat_1 * public_debt_df.shift(2))

# Column 3: CB financial independence interaction
panel_df['infl_hat X cb_fin_indep']  = melt(infl_hat_1 * cb_fin_indep_relative_df.shift(2))

## 8. Table 6 — Heterogeneity Regressions

Each cell below calls `run_heterogeneity_regression`, passing the interaction term
and the corresponding level variable. The function handles panel construction,
NaN dropping, and display automatically.

### Column 1: Wage Growth

Countries with lower prior wage growth should see a stronger inflation-crisis link,
since their households have less capacity to absorb real income erosion.
A negative interaction coefficient is the expected sign.

In [43]:
run_heterogeneity_regression(['infl_hat X wages', 'wages'])

,results_w_errors
const,0.055*** (0.011)
delta_inflation_hat,0.052*** (0.017)
infl_hat X wages,-0.001** (0.001)
wages,-0.001 (0.001)


### Column 2: Debt

We test three debt types simultaneously. The redistribution channel predicts that
the inflation-crisis link is stronger in countries with higher *household* debt
(proxied by mortgages), but not necessarily for corporate or public debt.
A positive interaction on mortgages is the key prediction.

In [44]:
run_heterogeneity_regression([
    'infl_hat X mortgages',     'mortgages',
    'infl_hat X corporate_debt','corporate_debt',
    'infl_hat X public_debt',   'public_debt',
])

,results_w_errors
const,-0.004 (0.034)
delta_inflation_hat,-0.012 (0.025)
infl_hat X mortgages,0.080*** (0.026)
mortgages,0.064 (0.048)
infl_hat X corporate_debt,0.012 (0.008)
corporate_debt,0.043 (0.032)
infl_hat X public_debt,0.020 (0.02)
public_debt,-0.024 (0.02)


### Column 3: Central Bank Financial Independence

Higher CB financial independence should anchor inflation expectations more firmly,
dampening second-round effects and reducing the crisis impact of inflationary shocks.
A negative interaction coefficient is the expected sign.

In [45]:
run_heterogeneity_regression(['infl_hat X cb_fin_indep', 'cb_fin_indep'])

,results_w_errors
const,0.054*** (0.012)
delta_inflation_hat,0.044** (0.017)
infl_hat X cb_fin_indep,-0.073* (0.044)
cb_fin_indep,0.021 (0.038)
